In [7]:
pip install curl_cffi pandas beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


In [9]:
import time
import json
import pandas as pd
from curl_cffi import requests as cffi_requests

#  Session setup with browser impersonation

session = cffi_requests.Session(impersonate="chrome120")
session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "en-US,en;q=0.9",
})


# Acquire Yahoo crumb (required for all query1.finance.yahoo.com calls)

def get_crumb() -> str:
    """
    Yahoo's API requires a 'crumb' token tied to a session cookie.
    Steps:
      1. Visit finance.yahoo.com to get tracking cookies.
      2. Call /v1/test/getcrumb to receive the token.
    """
    try:
        session.get("https://finance.yahoo.com/", timeout=15)
        time.sleep(1.5)  # mimic human behavior

        res = session.get(
            "https://query1.finance.yahoo.com/v1/test/getcrumb",
            timeout=10,
        )
        crumb = res.text.strip()

        if not crumb:
            raise RuntimeError(
                f"Empty crumb returned (status={res.status_code}). "
                "Yahoo may be rate-limiting this IP."
            )
        return crumb
    except Exception as e:
        raise RuntimeError(f"Failed to obtain crumb: {e}")


CRUMB = get_crumb()
print(f"Crumb obtained: {CRUMB}")

# 3. Fetch a single page of screener data

def fetch_page(scr_id: str, count: int = 25, offset: int = 0):
    """
    Fetch one page of results for a predefined Yahoo screener.
    Returns (records_list, total_count).
    """
    url = "https://query1.finance.yahoo.com/v1/finance/screener/predefined/saved"
    params = {
        "scrIds": scr_id,
        "count": count,
        "offset": offset,
        "formatted": "true",
        "lang": "en-US",
        "region": "US",
        "crumb": CRUMB,
    }

    r = session.get(url, params=params, timeout=15)
    r.raise_for_status()
    payload = r.json()

    finance = payload.get("finance", {})

    # Yahoo always includes an "error" key; it's None on success.
    # Only raise when it's a real error object.
    if finance.get("error"):
        raise RuntimeError(f"API error: {finance['error']}")

    results = finance.get("result") or []
    if not results:
        print("No 'result' block in response. Raw payload preview:")
        print(json.dumps(payload, indent=2)[:1500])
        return [], 0

    block = results[0]
    records = block.get("records", [])
    total = block.get("total", 0)

    # Fallback: Yahoo occasionally renames 'records'
    if not records and total > 0:
        print(f"'records' key empty. Keys in result[0]: {list(block.keys())}")
        for alt in ("rows", "data", "quotes"):
            if alt in block:
                records = block[alt]
                print(f"   → Using fallback key: '{alt}'")
                break

    return records, total

# Paginate through all results

SCREENER_ID = "most_actives"   # try: day_gainers, day_losers, most_shorted_stocks, ...
PAGE_SIZE = 25

all_records = []
offset = 0

print(f"Scraping screener: {SCREENER_ID}")
while True:
    records, total = fetch_page(SCREENER_ID, count=PAGE_SIZE, offset=offset)

    if not records:
        print("   No more records returned. Stopping.")
        break

    all_records.extend(records)
    print(f"   offset={offset:<5} got {len(records):>3} records "
          f"| total so far: {len(all_records)}/{total}")

    offset += PAGE_SIZE
    if offset >= total:
        break

    time.sleep(1)  # respectful delay between requests

print(f"Collected {len(all_records)} records total.")

# Build DataFrame and save CSVs

if not all_records:
    raise SystemExit("No records were returned. Check the debug output above.")

# Flatten nested dicts like {'raw': 123.45, 'fmt': '123.45'}
df = pd.json_normalize(all_records)

print(f"DataFrame shape: {df.shape}")
print(f"   Columns: {list(df.columns)[:15]}{' ...' if df.shape[1] > 15 else ''}")

# Save EVERYTHING first, so nothing is ever lost
df.to_csv("yahoo_screener_full.csv", index=False)
print("Full data saved → yahoo_screener_full.csv")

# Save a tidy subset using whichever preferred columns actually exist
preferred_columns = [
    "symbol",
    "shortName",
    "longName",
    "regularMarketPrice",
    "regularMarketChange",
    "regularMarketChangePercent",
    "regularMarketVolume",
    "marketCap",
    "regularMarketPreviousClose",
    "regularMarketOpen",
    "regularMarketDayHigh",
    "regularMarketDayLow",
    "fiftyTwoWeekHigh",
    "fiftyTwoWeekLow",
]

existing = [c for c in preferred_columns if c in df.columns]

if existing:
    curated = df[existing]
    curated.to_csv("yahoo_screener_curated.csv", index=False)
    print(f"Curated CSV saved → yahoo_screener_curated.csv")
    print(f"Columns: {existing}")
    print("Preview:")
    print(curated.head())
else:
    print("None of the preferred columns exist.")
    print("Inspect yahoo_screener_full.csv for actual field names.")
    print(f"Available columns: {list(df.columns)}")


print("Task 1 of Level 1 Completed by Omar ")

Crumb obtained: {"finance":{"result":null,"error":{"code":"Unauthorized","description":"Invalid Cookie"}}}
Scraping screener: most_actives
'records' key empty. Keys in result[0]: ['id', 'title', 'description', 'canonicalName', 'criteriaMeta', 'rawCriteria', 'start', 'count', 'total', 'quotes', 'useRecords', 'predefinedScr', 'versionId', 'creationDate', 'lastUpdated', 'isPremium', 'iconUrl']
   → Using fallback key: 'quotes'
   offset=0     got  25 records | total so far: 25/382
'records' key empty. Keys in result[0]: ['id', 'title', 'description', 'canonicalName', 'criteriaMeta', 'rawCriteria', 'start', 'count', 'total', 'quotes', 'useRecords', 'predefinedScr', 'versionId', 'creationDate', 'lastUpdated', 'isPremium', 'iconUrl']
   → Using fallback key: 'quotes'
   offset=25    got  25 records | total so far: 50/382
'records' key empty. Keys in result[0]: ['id', 'title', 'description', 'canonicalName', 'criteriaMeta', 'rawCriteria', 'start', 'count', 'total', 'quotes', 'useRecords', 'pr